In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import SimpleITK as sitk
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

# Tableau des 112 participants, avec les chemins IRM et PET (notebook 06).
base = pd.read_csv(C.BASE_112, dtype="string")
base["PTID"] = base["PTID"].str.strip().str.upper()
for col in ["chemin_irm", "chemin_pet"]:
    manquants = base[col].map(lambda p: not Path(str(p)).is_file()).sum()
    if manquants:
        raise FileNotFoundError(f"{manquants} fichier(s) absent(s) dans {col}.")

print("Participants a traiter :", len(base))

# Dossier de sortie (horodate, sur le disque externe).
SORTIE = C.sortie("features_image")
DOSSIER_NPY = SORTIE / "volumes_64"
DOSSIER_NPY.mkdir(parents=True, exist_ok=True)
print("Sortie :", SORTIE)

In [ ]:
# Fonction de pretraitement d'un volume : redimensionner + normaliser.
TAILLE = (64, 64, 64)

def preparer(chemin, modalite):
    img = sitk.ReadImage(str(chemin))

    # Reechantillonner vers une grille fixe 64x64x64.
    taille_orig = img.GetSize()
    espacement_orig = img.GetSpacing()
    espacement_cible = [
        espacement_orig[i] * taille_orig[i] / TAILLE[i] for i in range(3)
    ]
    reference = sitk.Image(TAILLE, img.GetPixelIDValue())
    reference.SetOrigin(img.GetOrigin())
    reference.SetDirection(img.GetDirection())
    reference.SetSpacing(espacement_cible)
    img = sitk.Resample(img, reference, sitk.Transform(),
                        sitk.sitkLinear, 0.0)

    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    cerveau = arr[arr > 0]  # ignorer le fond (voxels nuls)
    if cerveau.size == 0:
        raise ValueError("Volume vide apres reechantillonnage.")

    if modalite == "IRM":
        # Centrage-reduction sur les voxels du cerveau.
        arr = (arr - cerveau.mean()) / (cerveau.std() + 1e-6)
    else:
        # PET : division par l'intensite cerebrale moyenne (pseudo-SUVR).
        arr = arr / (cerveau.mean() + 1e-6)

    return arr

def caracteristiques(arr, prefixe):
    cerveau = arr[arr != 0]
    if cerveau.size == 0:
        cerveau = arr.ravel()
    p10, p50, p90 = np.percentile(cerveau, [10, 50, 90])
    return {
        f"{prefixe}_moyenne": float(cerveau.mean()),
        f"{prefixe}_ecart_type": float(cerveau.std()),
        f"{prefixe}_p10": float(p10),
        f"{prefixe}_p50": float(p50),
        f"{prefixe}_p90": float(p90),
    }

In [ ]:
# Boucle principale : une image a la fois, enregistrement immediat.
journal = SORTIE / "features_image.csv"
deja = pd.read_csv(journal, dtype="string") if journal.exists() else None
faits = set(deja["PTID"]) if deja is not None else set()

lignes = deja.to_dict("records") if deja is not None else []
erreurs = []

for n, ligne in enumerate(base.itertuples(index=False), start=1):
    if ligne.PTID in faits:
        continue

    enregistrement = {"PTID": ligne.PTID}
    try:
        for modalite, chemin in [("IRM", ligne.chemin_irm), ("PET", ligne.chemin_pet)]:
            arr = preparer(chemin, modalite)
            np.save(DOSSIER_NPY / f"{ligne.PTID}_{modalite}.npy", arr)
            enregistrement.update(caracteristiques(arr, modalite))
            del arr  # liberer la memoire tout de suite
        lignes.append(enregistrement)
    except Exception as e:
        erreurs.append({"PTID": ligne.PTID, "erreur": f"{type(e).__name__}: {e}"})

    # Sauvegarde reguliere pour pouvoir reprendre.
    if n % 10 == 0 or n == len(base):
        pd.DataFrame(lignes).to_csv(journal, index=False, encoding="utf-8-sig")
        print(f"{n}/{len(base)} traites", flush=True)

pd.DataFrame(lignes).to_csv(journal, index=False, encoding="utf-8-sig")
print("\nTermine. Participants avec caracteristiques :", len(lignes))
print("Erreurs :", len(erreurs))
if erreurs:
    display(pd.DataFrame(erreurs))

In [ ]:
# Verification rapide du resultat.
features = pd.read_csv(journal)
print("Tableau des caracteristiques d'image :", features.shape)
print("Volumes .npy enregistres :", len(list(DOSSIER_NPY.glob("*.npy"))),
      "(attendu : 2 x 112 = 224)")
display(features.head())
print("\nDossier a reutiliser au notebook 10 :", SORTIE.name)

In [ ]:
# --- Reprise : finir le pretraitement des images restantes ---
from pathlib import Path
import sys, numpy as np, pandas as pd, SimpleITK as sitk
from IPython.display import display

sys.path.append(r"..\configs")
from config_projet import C

TAILLE = (64, 64, 64)

def preparer(chemin, modalite):
    img = sitk.ReadImage(str(chemin))
    to, eo = img.GetSize(), img.GetSpacing()
    ec = [eo[i] * to[i] / TAILLE[i] for i in range(3)]
    ref = sitk.Image(TAILLE, img.GetPixelIDValue())
    ref.SetOrigin(img.GetOrigin()); ref.SetDirection(img.GetDirection()); ref.SetSpacing(ec)
    img = sitk.Resample(img, ref, sitk.Transform(), sitk.sitkLinear, 0.0)
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    cerveau = arr[arr > 0]
    if cerveau.size == 0:
        raise ValueError("Volume vide apres reechantillonnage.")
    if modalite == "IRM":
        arr = (arr - cerveau.mean()) / (cerveau.std() + 1e-6)
    else:
        arr = arr / (cerveau.mean() + 1e-6)
    return arr

def caracteristiques(arr, prefixe):
    c = arr[arr != 0]
    if c.size == 0: c = arr.ravel()
    p10, p50, p90 = np.percentile(c, [10, 50, 90])
    return {f"{prefixe}_moyenne": float(c.mean()), f"{prefixe}_ecart_type": float(c.std()),
            f"{prefixe}_p10": float(p10), f"{prefixe}_p50": float(p50), f"{prefixe}_p90": float(p90)}

# Reprendre le DERNIER dossier features_image_* (sans en recreer un).
SORTIE = sorted(C.COHORTE.glob("features_image_*"))[-1]
DOSSIER_NPY = SORTIE / "volumes_64"; DOSSIER_NPY.mkdir(parents=True, exist_ok=True)
journal = SORTIE / "features_image.csv"
print("Reprise dans :", SORTIE.name)

base = pd.read_csv(C.BASE_112, dtype="string")
base["PTID"] = base["PTID"].str.strip().str.upper()

deja = pd.read_csv(journal, dtype="string") if journal.exists() else None
faits = set(deja["PTID"]) if deja is not None else set()
lignes = deja.to_dict("records") if deja is not None else []
restants = base.loc[~base["PTID"].isin(faits)]
print("Deja traites :", len(faits), "/ 112 | restants :", len(restants))

erreurs = []
for n, ligne in enumerate(restants.itertuples(index=False), start=1):
    enr = {"PTID": ligne.PTID}
    try:
        for modalite, chemin in [("IRM", ligne.chemin_irm), ("PET", ligne.chemin_pet)]:
            arr = preparer(chemin, modalite)
            np.save(DOSSIER_NPY / f"{ligne.PTID}_{modalite}.npy", arr)
            enr.update(caracteristiques(arr, modalite)); del arr
        lignes.append(enr)
    except Exception as e:
        erreurs.append({"PTID": ligne.PTID, "erreur": f"{type(e).__name__}: {e}"})
    pd.DataFrame(lignes).to_csv(journal, index=False, encoding="utf-8-sig")  # sauvegarde a chaque image
    print(f"{len(lignes)}/112 | reste {len(restants)-n}", flush=True)

print("\nTermine. Caracteristiques :", len(lignes), "| Erreurs :", len(erreurs))
print("Volumes .npy :", len(list(DOSSIER_NPY.glob('*.npy'))), "(attendu 224)")
if erreurs: display(pd.DataFrame(erreurs))